The objective of this notebook is to transform historical retail sales into predictive variables suitable for machine learning.

Feature engineering is one of the most important stages of the forecasting pipeline because it captures historical demand behaviour, seasonality, pricing effects, and calendar influences that are not directly available in the raw data.
- Create lag features
- Create rolling demand statistics
- Engineer calendar features
- Engineer pricing features
- Remove unusable variables
- Produce the final modeling dataset


In [5]:
from src.utils.project_setup import *

from src.data.loader import load_master_data

import pandas as pd
import numpy as np

In [6]:
master = load_master_data()

master.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN


In [7]:
#sort data
master = master.sort_values(
    ["store_id", "item_id", "date"]
)

In [8]:
# Convert the date column to datetime
master["date"] = pd.to_datetime(master["date"], format="%Y-%m-%d")

In [9]:
#creating time features
master["year"] = master["date"].dt.year
master["month"] = master["date"].dt.month
master["week"] = master["date"].dt.isocalendar().week
master["day"] = master["date"].dt.day
master["dayofweek"] = master["date"].dt.dayofweek
master["quarter"] = master["date"].dt.quarter
master["is_weekend"] = (
    master["dayofweek"] >= 5
).astype(int)

In [10]:
#creating lag features
group = ["store_id", "item_id"]

master["lag_7"] = (
    master.groupby(group)["sales"]
    .shift(7)
)

master["lag_14"] = (
    master.groupby(group)["sales"]
    .shift(14)
)

master["lag_28"] = (
    master.groupby(group)["sales"]
    .shift(28)
)

In [11]:
#creating rolling statistics
master["rolling_mean_7"] = (
    master.groupby(group)["sales"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

master["rolling_mean_28"] = (
    master.groupby(group)["sales"]
    .transform(
        lambda x: x.shift(1).rolling(28).mean()
    )
)

master["rolling_std_28"] = (
    master.groupby(group)["sales"]
    .transform(
        lambda x: x.shift(1).rolling(28).std()
    )
)

In [12]:
#creating price features
master["price_change"] = (
    master.groupby(group)["sell_price"]
    .pct_change()
)

master["price_diff"] = (
    master.groupby(group)["sell_price"]
    .diff()
)

In [13]:
#creating calender features
master["is_event"] = (
    master["event_name_1"]
    .notna()
    .astype(int)
)

In [14]:
#demand  trend, rolling max
master["rolling_max_28"] = (
    master.groupby(group)["sales"]
    .transform(
        lambda x: x.shift(1).rolling(28).max()
    )
)

In [15]:
#demand trend, rolling minimum
master["rolling_min_28"] = (
    master.groupby(group)["sales"]
    .transform(
        lambda x: x.shift(1).rolling(28).min()
    )
)

In [16]:
# validating features
sample = (
    master[
        master["item_id"] == master["item_id"].iloc[0]
    ]
    .head(40)
)

sample[
    [
        "date",
        "sales",
        "lag_7",
        "lag_28",
        "rolling_mean_28"
    ]
]

,date,sales,lag_7,lag_28,rolling_mean_28
1612,2011-01-29,3,NaN,NaN,NaN
32102,2011-01-30,0,NaN,NaN,NaN
62592,2011-01-31,0,NaN,NaN,NaN
93082,2011-02-01,1,NaN,NaN,NaN
123572,2011-02-02,4,NaN,NaN,NaN
154062,2011-02-03,2,NaN,NaN,NaN
184552,2011-02-04,0,NaN,NaN,NaN
215042,2011-02-05,2,3.0,NaN,NaN
245532,2011-02-06,0,0.0,NaN,NaN
276022,2011-02-07,0,0.0,NaN,NaN


In [17]:
#reviewing engineered features
feature_cols = [
    "sales",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_28",
    "rolling_max_28",
    "rolling_min_28",
    "price_change",
    "price_diff",
    "year",
    "month",
    "week",
    "dayofweek",
    "is_weekend",
    "is_event"
]

master[feature_cols].head(15)

,sales,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_28,rolling_max_28,rolling_min_28,price_change,price_diff,year,month,week,dayofweek,is_weekend,is_event
1612,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011,1,4,5,1,0
32102,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2011,1,4,6,1,0
62592,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2011,1,5,0,0,0
93082,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2011,2,5,1,0,0
123572,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2011,2,5,2,0,0
154062,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2011,2,5,3,0,0
184552,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2011,2,5,4,0,0
215042,2,3.0,NaN,NaN,1.428571,NaN,NaN,NaN,NaN,0.0,0.0,2011,2,5,5,1,0
245532,0,0.0,NaN,NaN,1.285714,NaN,NaN,NaN,NaN,0.0,0.0,2011,2,5,6,1,1
276022,0,0.0,NaN,NaN,1.285714,NaN,NaN,NaN,NaN,0.0,0.0,2011,2,6,0,0,0


In [18]:
missing = (
    master[feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing

price_diff         12329903
price_change       12329903
rolling_min_28       853720
lag_28               853720
rolling_max_28       853720
rolling_std_28       853720
rolling_mean_28      853720
lag_14               426860
lag_7                213430
rolling_mean_7       213430
sales                     0
year                      0
month                     0
week                      0
dayofweek                 0
is_weekend                0
is_event                  0
dtype: int64

In [19]:
missing_pct = (
    master[feature_cols]
    .isna()
    .mean()
    * 100
).round(2)

missing_pct.sort_values(ascending=False)

price_diff         21.14
price_change       21.14
rolling_min_28      1.46
lag_28              1.46
rolling_max_28      1.46
rolling_std_28      1.46
rolling_mean_28     1.46
lag_14              0.73
lag_7               0.37
rolling_mean_7      0.37
sales               0.00
year                0.00
month               0.00
week                0.00
dayofweek           0.00
is_weekend          0.00
is_event            0.00
dtype: float64

In [20]:
master[feature_cols].dtypes

sales                int64
lag_7              float64
lag_14             float64
lag_28             float64
rolling_mean_7     float64
rolling_mean_28    float64
rolling_std_28     float64
rolling_max_28     float64
rolling_min_28     float64
price_change       float64
price_diff         float64
year                 int32
month                int32
week                UInt32
dayofweek            int32
is_weekend           int64
is_event             int64
dtype: object

In [21]:
master.to_parquet(
    "../data/processed/model_dataset.parquet",
    index=False
)

## Completed

- Engineered temporal features.
- Created lag demand variables.
- Created rolling demand statistics.
- Generated pricing features.
- Added calendar indicators.
- Validated feature quality.
- Saved the final modeling dataset.

The dataset is now ready for machine learning model development.